# Haematopoietic reference projection

Run this notebook from the repository root and execute cells from top to bottom. All inputs use repository-relative paths. Generated files are written below `data/processed/` or `results/`. Outputs and execution counters are cleared in the version-controlled copy.


In [ ]:
# Step 0: resolve repository-relative paths.
project_root <- normalizePath(Sys.getenv("BMO_PROJECT_ROOT", unset = "."), winslash = "/", mustWork = TRUE)
dir.create(file.path(project_root, "data", "processed"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "figures"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(project_root, "results", "tables"), recursive = TRUE, showWarnings = FALSE)

library(Seurat)
library(SeuratObject)
library(tidyverse)


<b><font size=5 color=pink >Step 1: load the published 2023 organoid dataset</font></b>


In [ ]:
load(file.path(project_root, "data", "reference", "organoids23", "Organoids23.RData"))
DimPlot(Organoids23, label = TRUE, group.by = "celltype")


In [ ]:
table(Organoids23@meta.data$celltype)


In [ ]:
library(dplyr)

Organoids23$compartment <- dplyr::case_when(
  Organoids23$celltype %in% c(
    "Endothelium",
    "Fibroblast",
    "MSC"
  ) ~ "Stromal",

  Organoids23$celltype %in% c(
    "Erythroid",
    "HSPC",
    "Megakaryocyte",
    "Monocyte",
    "Myeloid Progenitor"
  ) ~ "Haematopoietic",

  TRUE ~ NA_character_
)

table(Organoids23$compartment, useNA = "ifany")

#Organoids23_Stromal <- subset(
  #Organoids23,
  #subset = compartment == "Stromal"
#)

Organoids23_Haematopoietic <- subset(
  Organoids23,
  subset = compartment == "Haematopoietic"
)

#table(Organoids23_Stromal$celltype)
table(Organoids23_Haematopoietic$celltype)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$orig.ident)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$data_set)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$sampleID)


In [ ]:
Organoids23_Haematopoietic <- NormalizeData(Organoids23_Haematopoietic)
Organoids23_Haematopoietic <- FindVariableFeatures(Organoids23_Haematopoietic, selection.method = "vst", nfeatures = 2000)

all.genes <- rownames(Organoids23_Haematopoietic)
Organoids23_Haematopoietic <- ScaleData(Organoids23_Haematopoietic, features = all.genes)

Organoids23_Haematopoietic <- RunPCA(Organoids23_Haematopoietic, features = VariableFeatures(object = Organoids23_Haematopoietic))

ElbowPlot(Organoids23_Haematopoietic, ndims = 30)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$celltype)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$annotations)


In [ ]:
Organoids23_Haematopoietic <- FindNeighbors(Organoids23_Haematopoietic, dims = 1:10)
Organoids23_Haematopoietic <- FindClusters(Organoids23_Haematopoietic, resolution = 3)

Organoids23_Haematopoietic <- RunUMAP(Organoids23_Haematopoietic, dims = 1:10, return.model = TRUE)
Organoids23_Haematopoietic <- RunTSNE(Organoids23_Haematopoietic, dims = 1:10, return.model = TRUE)

p1 <- DimPlot(Organoids23_Haematopoietic, reduction = "tsne", group.by = "celltype", label = TRUE) + ggtitle("t-SNE")
p2 <- DimPlot(Organoids23_Haematopoietic, reduction = "umap", group.by = "celltype", label = TRUE) + ggtitle("UMAP")

p1/p2


In [ ]:
saveRDS(Organoids23_Haematopoietic, file = file.path(project_root, "data", "processed", "Organoids23_Haematopoietic.rds"))


In [ ]:
Organoids23_Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "Organoids23_Haematopoietic.rds"))


<b><font size=5 color=pink >Step 2: load BMO haematopoietic cells</font></b>


In [ ]:
### ============================================
### ============================================
Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "Haematopoietic.rds"))

Haematopoietic@meta.data$group <- ""
Haematopoietic@meta.data$group[Haematopoietic@meta.data$orig.ident %in% c("Dynamic.25d.1", "Dynamic.25d.2", "Dynamic.25d.3")] <- "Dynamic.25d"
Haematopoietic@meta.data$group[Haematopoietic@meta.data$orig.ident %in% c("Dynamic.31d")] <- "Dynamic.31d"
Haematopoietic@meta.data$group[Haematopoietic@meta.data$orig.ident %in% c("Static.25d.1", "Static.25d.2", "Static.25d.3")] <- "Static.25d"
table(Haematopoietic@meta.data$group)

Haematopoietic@meta.data$celltype <- ""
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("17")] <- "HSC"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("0","1","4","5","6","7","10","12","13")] <- "Erythroid"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("14")] <- "Mast"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("19")] <- "DC"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("3", "9")] <- "Macrophage"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("2", "11", "8")] <- "Monocyte"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("15")] <- "Basophil"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("16")] <- "Eosinophil"
Haematopoietic@meta.data$celltype[Haematopoietic@meta.data$seurat_clusters %in% c("18")] <- "Neutrophil"

mk_barcodes <- WhichCells(Haematopoietic, expression = PPBP > 1 | PF4 > 1)
valid_mk <- intersect(mk_barcodes, colnames(Haematopoietic))
Haematopoietic@meta.data[valid_mk, "celltype"] <- "Megakaryocyte"
table(Haematopoietic$celltype)

custom_cols_hema <- c(
  "HSC" = "#555555", "Megakaryocyte" = "#FFD92F", "Erythroid" = "#E41A1C",
  "Neutrophil" = "#1F78B4", "Eosinophil" = "#A6CEE3", "Basophil" = "#33A02C",
  "Mast" = "#B2DF8A", "Monocyte" = "#FDBF6F", "Macrophage" = "#B15928", "DC" = "#CAB2D6"
)
Haematopoietic$celltype <- factor(Haematopoietic$celltype, levels = names(custom_cols_hema))

DimPlot(Haematopoietic, reduction = "tsne", group.by = "celltype",
        label = TRUE, cols = custom_cols_hema, pt.size = 0.8) +
  theme(aspect.ratio = 1)


<b><font size=5 color=pink >Step 3: load adult bone marrow</font></b>


In [ ]:
#Adult_BM_Stromal <- readRDS(file.path(project_root, "data", "processed", "Adult_BM_Stromal.rds"))


In [ ]:
Adult_BM_Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "Adult_BM_Haematopoietic.rds"))


In [ ]:
Adult_BM_Haematopoietic[["umap"]]@misc$model


In [ ]:
table(Adult_BM_Haematopoietic@meta.data$cluster_anno_l2)


In [ ]:
DimPlot(Adult_BM_Haematopoietic, reduction = "umap", group.by = "cluster_anno_l2", label = TRUE) + NoLegend()


In [ ]:
table(Adult_BM_Haematopoietic@meta.data$cluster_anno_l2)


In [ ]:
library(Seurat)
library(ggplot2)

# ============================================================
# ============================================================

haem_levels <- c(
  "Plasma Cell",
  "RBC",
  "Late Erythroid",
  "Erythroblast",
  "MEP",
  "Megakaryocyte",
  "HSC",
  "MPP",
  "Cycling HSPC",
  "GMP",
  "CLP",
  "Early Myeloid Progenitor",
  "Late Myeloid",
  "Neutrophil",
  "Monocyte",
  "Macrophages",
  "Ba/Eo/Ma",
  "pDC",
  "Cycling DCs",
  "Pre-Pro B",
  "Pro-B",
  "Pre-B",
  "Mature B",
  "CD4+ T-Cell",
  "CD8+ T-Cell"
)

Adult_BM_Haematopoietic$cluster_anno_l2 <- factor(
  Adult_BM_Haematopoietic$cluster_anno_l2,
  levels = haem_levels
)

# ============================================================
# ============================================================

haem_cols_position <- c(
  "Plasma Cell"              = "#B5E48C",

  "RBC"                      = "#FFD166",
  "Late Erythroid"           = "#F4A261",
  "Erythroblast"             = "#E76F51",
  "MEP"                      = "#8D5A97",
  "Megakaryocyte"            = "#A6761D",

  "HSC"                      = "#D62828",
  "MPP"                      = "#F77F00",
  "Cycling HSPC"             = "#BC6C25",
  "GMP"                      = "#7ac2deff",
  "CLP"                      = "#6C584C",

  "Early Myeloid Progenitor" = "#80CDC1",
  "Late Myeloid"             = "#018571",
  "Neutrophil"               = "#4DBBD5",
  "Monocyte"                 = "#2B6CB0",
  "Macrophages"              = "#003049",

  "Ba/Eo/Ma"                 = "#F6BD60",
  "pDC"                      = "#5E3C99",
  "Cycling DCs"              = "#B39DDB",

  "Pre-Pro B"                = "#A7C957",
  "Pro-B"                    = "#70AD47",
  "Pre-B"                    = "#1B9E77",
  "Mature B"                 = "#006D77",

  "CD4+ T-Cell"              = "#C51B7D",
  "CD8+ T-Cell"              = "#F768A1"
)


In [ ]:
# ============================================================
# ============================================================

p_adult_bm_haem <- DimPlot(
  Adult_BM_Haematopoietic,
  reduction = "umap",
  group.by = "cluster_anno_l2",
  label = TRUE,
  repel = TRUE,
  label.size = 3,
  pt.size = 0.25,
  cols = haem_cols_position
) +
  NoAxes() +
  NoLegend() +
  labs(
    title = "ABM haematopoietic cell types"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    )
  )

p_adult_bm_haem


In [ ]:
# Figures are written explicitly to results/figures; no working-directory change is required.


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "1_ABM_Haematopoietic_celltype_UMAP.pdf"),
  plot = p_adult_bm_haem,
  width = 5,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 4: load fetal bone marrow</font></b>


In [ ]:
#FBM_Stromal <- readRDS(file.path(project_root, "data", "processed", "FBM_Stromal.rds"))


In [ ]:
FBM_Haematopoietic <- readRDS(file.path(project_root, "data", "processed", "FBM_Haematopoietic.rds"))


In [ ]:
FBM_Haematopoietic[["umap"]]@misc$model


In [ ]:
table(FBM_Haematopoietic@meta.data$broad_fig1_cell.labels)


In [ ]:
DimPlot(FBM_Haematopoietic, reduction = "umap", group.by = "broad_fig1_cell.labels", label = TRUE) + NoLegend()


In [ ]:
library(Seurat)
library(ggplot2)

# ============================================================
# ============================================================

fbm_haem_levels <- c(
  "HSC/MPP and pro",
  "erythroid",
  "MK",
  "B_lineage",
  "DC",
  "eo/baso/mast",
  "neutrophil",
  "monocyte",
  "T_NK"
)

FBM_Haematopoietic$broad_fig1_cell.labels <- factor(
  FBM_Haematopoietic$broad_fig1_cell.labels,
  levels = fbm_haem_levels
)

# ============================================================
# ============================================================

fbm_haem_cols <- c(
  "HSC/MPP and pro" = "#F6BD60",
  "erythroid"       = "#D62828",
  "MK"              = "#8C510A",
  "B_lineage"       = "#1B9E77",
  "DC"              = "#577590",
  "eo/baso/mast"    = "#F6BD60",
  "neutrophil"      = "#4DBBD5",
  "monocyte"        = "#2B6CB0",
  "T_NK"            = "#E76F51"
)

# ============================================================
# ============================================================

p_fbm_haem <- DimPlot(
  FBM_Haematopoietic,
  reduction = "umap",
  group.by = "broad_fig1_cell.labels",
  label = TRUE,
  repel = TRUE,
  label.size = 4,
  pt.size = 0.18,
  cols = fbm_haem_cols
) +
  NoAxes() +
  NoLegend() +
  labs(
    title = "FBM haematopoietic cell types"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    )
  )

p_fbm_haem


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "2_FBM_Haematopoietic_celltype_UMAP.pdf"),
  plot = p_fbm_haem,
  width = 5,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 5: map DBMOs to adult bone marrow</font></b>


In [ ]:
anchors.ABM.DBMOs <- FindTransferAnchors(
  reference = Adult_BM_Haematopoietic,
  query = Haematopoietic,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Haematopoietic <- MapQuery(
  anchorset = anchors.ABM.DBMOs,
  reference = Adult_BM_Haematopoietic,
  query = Haematopoietic,
  refdata = list(ABM = "cluster_anno_l2"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Haematopoietic@meta.data$predicted.ABM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(Adult_BM_Haematopoietic@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Haematopoietic@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Haematopoietic@meta.data$predicted.ABM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "DBMOs Haematopoietic Projection onto ABM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "3_DBMOs_Haematopoietic_Projection_onto_ABM_UMAP.pdf"),
  plot = p_beautiful,
  width = 6.3,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 5: map published organoids to adult bone marrow</font></b>


In [ ]:
anchors.ABM.23BMO <- FindTransferAnchors(
  reference = Adult_BM_Haematopoietic,
  query = Organoids23_Haematopoietic,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Organoids23_Haematopoietic <- MapQuery(
  anchorset = anchors.ABM.23BMO,
  reference = Adult_BM_Haematopoietic,
  query = Organoids23_Haematopoietic,
  refdata = list(ABM = "cluster_anno_l2"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$predicted.ABM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(Adult_BM_Haematopoietic@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Organoids23_Haematopoietic@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Organoids23_Haematopoietic@meta.data$predicted.ABM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "mBMO Haematopoietic Projection onto ABM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "4_mBMO_Haematopoietic_Projection_onto_ABM_UMAP.pdf"),
  plot = p_beautiful,
  width = 6.3,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 6: map DBMOs to fetal bone marrow</font></b>


In [ ]:
anchors.FBM.DBMOs <- FindTransferAnchors(
  reference = FBM_Haematopoietic,
  query = Haematopoietic,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Haematopoietic <- MapQuery(
  anchorset = anchors.FBM.DBMOs,
  reference = FBM_Haematopoietic,
  query = Haematopoietic,
  refdata = list(FBM = "broad_fig1_cell.labels"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Haematopoietic@meta.data$predicted.FBM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(FBM_Haematopoietic@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Haematopoietic@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Haematopoietic@meta.data$predicted.FBM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "DBMOs Haematopoietic Projection onto FBM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "5_DBMOs_Haematopoietic_Projection_onto_FBM_UMAP.pdf"),
  plot = p_beautiful,
  width = 6.3,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 6: map published organoids to fetal bone marrow</font></b>


In [ ]:
anchors.FBM.BMO23 <- FindTransferAnchors(
  reference = FBM_Haematopoietic,
  query = Organoids23_Haematopoietic,
  normalization.method = "LogNormalize",
  reference.assay = "RNA",
  query.assay = "RNA",
  dims = 1:10,
  reference.reduction = "pca"
)


In [ ]:
Organoids23_Haematopoietic <- MapQuery(
  anchorset = anchors.FBM.BMO23,
  reference = FBM_Haematopoietic,
  query = Organoids23_Haematopoietic,
  refdata = list(FBM = "broad_fig1_cell.labels"),
  reference.reduction = "pca",
  reduction.model = "umap"
)


In [ ]:
table(Organoids23_Haematopoietic@meta.data$predicted.FBM)


In [ ]:
# ==========================================
# ==========================================
ref_data <- as.data.frame(FBM_Haematopoietic@reductions$umap@cell.embeddings)
colnames(ref_data)[1:2] <- c("UMAP1", "UMAP2")

query_data <- as.data.frame(Organoids23_Haematopoietic@reductions$ref.umap@cell.embeddings)
colnames(query_data)[1:2] <- c("UMAP1", "UMAP2")
query_data$score <- Organoids23_Haematopoietic@meta.data$predicted.FBM.score

p_beautiful <- ggplot() +
  geom_point(data = ref_data, aes(x = UMAP1, y = UMAP2),
             color = "#e0e0e0", size = 0.5, alpha = 0.6) +

  geom_point(data = query_data, aes(x = UMAP1, y = UMAP2, color = score),
             size = 0.8) +

  scale_color_gradientn(
    colors = c("#071e3d", "#27a09d", "#f0e68c", "#a81c07"),
    name = "Projection Score"
  ) +
  theme_classic(base_size = 14) +
  labs(
    title = "Projection onto Original Adult BM UMAP",
    x = "UMAP_1",
    y = "UMAP_2"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 16
    ),
    legend.position = "right"
  )

p_beautiful


In [ ]:
ggsave(
  filename = file.path(project_root, "results", "figures", "6_mBMO_Haematopoietic_Projection_onto_FBM_UMAP.pdf"),
  plot = p_beautiful,
  width = 6.3,
  height = 5,
  units = "in"
)


<b><font size=5 color=pink >Step 7: reproduce the final comparison figure</font></b>


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)

# ============================================================
# ============================================================

score_df_Organoids23 <- Organoids23_Haematopoietic@meta.data %>%
  dplyr::select(predicted.ABM.score, predicted.FBM.score) %>%
  dplyr::rename(
    ABM = predicted.ABM.score,
    FBM = predicted.FBM.score
  ) %>%
  tidyr::pivot_longer(
    cols = c(ABM, FBM),
    names_to = "Reference",
    values_to = "Score"
  ) %>%
  dplyr::mutate(
    Dataset = "BMO-2023"
  )

# ============================================================
# ============================================================

score_df_DBMOs <- Haematopoietic@meta.data %>%
  dplyr::select(predicted.ABM.score, predicted.FBM.score) %>%
  dplyr::rename(
    ABM = predicted.ABM.score,
    FBM = predicted.FBM.score
  ) %>%
  tidyr::pivot_longer(
    cols = c(ABM, FBM),
    names_to = "Reference",
    values_to = "Score"
  ) %>%
  dplyr::mutate(
    Dataset = "DBMOs"
  )

# ============================================================
# ============================================================

plot_df <- dplyr::bind_rows(
  score_df_Organoids23,
  score_df_DBMOs
)

table(plot_df$Dataset, plot_df$Reference)
summary(plot_df$Score)


In [ ]:
# ============================================================
# ============================================================

p_score_density <- ggplot(
  plot_df,
  aes(x = Score, fill = Reference, color = Reference)
) +
  geom_density(
    alpha = 0.35,
    linewidth = 1,
    adjust = 1.5
  ) +
  facet_wrap(~ Dataset, nrow = 1) +
  coord_cartesian(xlim = c(0.1, 1.0)) +
  scale_fill_manual(
    values = c(
      "ABM" = "#d93025",
      "FBM" = "#1a73e8"
    )
  ) +
  scale_color_manual(
    values = c(
      "ABM" = "#d93025",
      "FBM" = "#1a73e8"
    )
  ) +
  theme_bw() +
  labs(
    title = "Projection mapping score to ABM and FBM",
    x = "Projection mapping score",
    y = "Cell density",
    fill = "Reference",
    color = "Reference"
  ) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 14,
      color = "black"
    ),
    strip.text = element_text(
      face = "bold",
      size = 12,
      color = "black"
    ),
    axis.text = element_text(color = "black"),
    axis.title = element_text(color = "black"),
    legend.position = "top",
    legend.title = element_text(color = "black"),
    legend.text = element_text(color = "black"),
    panel.grid.major = element_line(color = "#f0f0f0"),
    panel.grid.minor = element_blank()
  )

p_score_density


In [ ]:
library(ggplot2)
library(dplyr)

# ============================================================
# ============================================================

median_df <- plot_df %>%
  dplyr::group_by(Dataset, Reference) %>%
  dplyr::summarise(
    median_score = median(Score, na.rm = TRUE),
    .groups = "drop"
  )

p_score_density <- ggplot(
  plot_df,
  aes(x = Score, fill = Reference, color = Reference)
) +
  geom_density(
    alpha = 0.28,
    linewidth = 1.1,
    adjust = 1.4
  ) +

  geom_vline(
    data = median_df,
    aes(xintercept = median_score, color = Reference),
    linetype = "dashed",
    linewidth = 0.7,
    show.legend = FALSE
  ) +

  facet_wrap(~ Dataset, nrow = 1) +

  scale_x_continuous(
    limits = c(0.1, 1.0),
    breaks = seq(0.1, 1.0, by = 0.1),
    expand = c(0.01, 0.01)
  ) +

  scale_fill_manual(
    values = c(
      "ABM" = "#D73027",
      "FBM" = "#4575B4"
    )
  ) +
  scale_color_manual(
    values = c(
      "ABM" = "#D73027",
      "FBM" = "#4575B4"
    )
  ) +

  labs(
    title = "Projection mapping score to ABM and FBM",
    x = "Projection mapping score",
    y = "Cell density",
    fill = NULL,
    color = NULL
  ) +

  theme_classic(base_size = 13) +
  theme(
    plot.title = element_text(
      hjust = 0.5,
      face = "bold",
      size = 15,
      color = "black"
    ),
    strip.background = element_rect(
      fill = "#F2F2F2",
      color = NA
    ),
    strip.text = element_text(
      face = "bold",
      size = 12,
      color = "black"
    ),
    axis.title = element_text(
      size = 12,
      color = "black"
    ),
    axis.text = element_text(
      size = 10,
      color = "black"
    ),
    axis.line = element_line(
      color = "black",
      linewidth = 0.5
    ),
    axis.ticks = element_line(
      color = "black",
      linewidth = 0.4
    ),
    legend.position = "top",
    legend.text = element_text(
      size = 11,
      color = "black"
    ),
    panel.spacing = unit(1.2, "lines")
  )

p_score_density


In [ ]:
outdir <- file.path(project_root, "results", "figures", "Haematopoietic")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

ggsave(
  filename = file.path(outdir, "7_Haematopoietic_projection_score_density_BMO2023_DBMOs_optimized.pdf"),
  plot = p_score_density,
  width = 6,
  height = 4.2,
  units = "in",
  device = "pdf"
)
